In [1]:
# First, let's see what we can import
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp


def create_test_data(
    n_genes=500, n_cells=200, n_empty=1000, contamination=0.1, seed=42
):
    """Create simple synthetic data for testing."""
    np.random.seed(seed)

    # Create true expression for 5 cell types
    n_types = 5
    cells_per_type = n_cells // n_types
    true_counts = np.zeros((n_genes, n_cells))

    # Each cell type expresses different genes
    for cell_type in range(n_types):
        start_cell = cell_type * cells_per_type
        end_cell = min(start_cell + cells_per_type, n_cells)

        # High expression for marker genes
        marker_genes = range(cell_type * 20, (cell_type + 1) * 20)
        for cell in range(start_cell, end_cell):
            true_counts[list(marker_genes), cell] = np.random.poisson(20, 20)
            # Low background
            other_genes = list(set(range(n_genes)) - set(marker_genes))
            true_counts[other_genes, cell] = np.random.poisson(0.5, len(other_genes))

    # Create soup profile (average of all cells)
    soup_profile = np.mean(true_counts, axis=1)
    soup_profile = soup_profile / soup_profile.sum()

    # Add contamination
    observed = true_counts.copy()
    for cell in range(n_cells):
        cell_total = true_counts[:, cell].sum()
        contam_counts = np.random.poisson(soup_profile * cell_total * contamination)
        observed[:, cell] += contam_counts

    # Create empty droplets
    empty_counts = np.zeros((n_genes, n_empty))
    for i in range(n_empty):
        n_umis = np.random.poisson(10)  # Low UMI counts
        if n_umis > 0:
            empty_counts[:, i] = np.random.multinomial(n_umis, soup_profile)

    # Combine for raw matrix
    raw = np.hstack([empty_counts, observed])

    return {
        "raw": sp.csr_matrix(raw),
        "filtered": sp.csr_matrix(observed),
        "true": sp.csr_matrix(true_counts),
        "genes": [f"Gene_{i:04d}" for i in range(n_genes)],
        "cells": [f"Cell_{i:04d}" for i in range(n_cells)],
        "contamination": contamination,
        "cell_types": np.repeat(range(n_types), cells_per_type)[:n_cells],
    }


# Create test data
data = create_test_data(n_genes=300, n_cells=100, contamination=0.15, seed=42)
print(f"Created data: {data['raw'].shape[0]} genes, {data['raw'].shape[1]} droplets")
print(f"Filtered: {data['filtered'].shape[1]} cells")
print(f"True contamination: {data['contamination']:.1%}")

Created data: 300 genes, 1100 droplets
Filtered: 100 cells
True contamination: 15.0%


In [2]:
# Clean imports now
import anndata as adata_module

print("✓ Successfully imported decontx functions")

✓ Successfully imported decontx functions


In [3]:
# Create AnnData object from your test data
adata = adata_module.AnnData(
    X=data["filtered"].T,  # AnnData expects cells x genes
    var=pd.DataFrame(index=data["genes"]),
    obs=pd.DataFrame(index=data["cells"]),
)

# Add cell type info
adata.obs["cell_type"] = data["cell_types"]

print(f"Created AnnData: {adata.shape} (cells x genes)")
print(f"Cell types: {adata.obs['cell_type'].unique()}")
print(f"Data type: {type(adata.X)}")

Created AnnData: (100, 300) (cells x genes)
Cell types: [0 1 2 3 4]
Data type: <class 'scipy.sparse._csc.csc_matrix'>


### PBMC3k testing - prepare with scanpy, then python vs R implementation

In [ ]:
import os
import tarfile
from pathlib import Path

import pandas as pd
from scipy.io import mmread


def load_10x_data_fixed(raw_tar_path, filtered_tar_path, extract_dir="./temp_10x/"):
    """
    Fixed loader for 10X Genomics data from tar.gz files.
    Properly handles matrix orientation - 10X matrices are stored as features x barcodes.
    """
    # Create extraction directory
    Path(extract_dir).mkdir(exist_ok=True)

    def find_matrix_files(base_dir):
        """Recursively find matrix.mtx, features.tsv/genes.tsv, barcodes.tsv"""
        matrix_file = None
        features_file = None
        barcodes_file = None
        for root, _dirs, files in os.walk(base_dir):
            for file in files:
                file_path = os.path.join(root, file)
                if file == "matrix.mtx":
                    matrix_file = file_path
                elif file in ["features.tsv", "genes.tsv"]:
                    features_file = file_path
                elif file == "barcodes.tsv":
                    barcodes_file = file_path
        return matrix_file, features_file, barcodes_file

    def extract_and_load_matrix(tar_path, matrix_name):
        """Extract tar and load sparse matrix with correct orientation"""
        # Extract to subdirectory to avoid conflicts
        extract_subdir = os.path.join(extract_dir, matrix_name)
        Path(extract_subdir).mkdir(exist_ok=True)
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(extract_subdir)
        print(f"Loading {matrix_name} from {extract_subdir}")
        # Find matrix files recursively
        matrix_file, features_file, barcodes_file = find_matrix_files(extract_subdir)
        if not all([matrix_file, features_file, barcodes_file]):
            print(f"Missing files in {extract_subdir}:")
            print(f"  matrix.mtx: {matrix_file}")
            print(f"  features/genes.tsv: {features_file}")
            print(f"  barcodes.tsv: {barcodes_file}")
            raise FileNotFoundError("Required 10X files not found")
        # Load sparse matrix - 10X format is features x barcodes (genes x cells)
        matrix = mmread(
            matrix_file
        ).tocsr()  # NO transpose - already correct orientation
        # Load gene names
        genes_df = pd.read_csv(features_file, sep="\t", header=None)
        if genes_df.shape[1] >= 2:
            gene_names = genes_df.iloc[:, 1].values  # Gene symbols (column 2)
        else:
            gene_names = genes_df.iloc[:, 0].values  # Gene IDs (column 1)
        # Load barcodes
        barcodes = pd.read_csv(barcodes_file, sep="\t", header=None).iloc[:, 0].values
        print(f"  Shape: {matrix.shape} (genes x cells)")
        print(f"  Genes: {len(gene_names)}")
        print(f"  Barcodes: {len(barcodes)}")
        # Validation - matrix should be genes x barcodes
        if matrix.shape[0] != len(gene_names):
            print(
                f"WARNING: Matrix rows ({matrix.shape[0]}) != gene names ({len(gene_names)})"
            )
        if matrix.shape[1] != len(barcodes):
            print(
                f"WARNING: Matrix cols ({matrix.shape[1]}) != barcodes ({len(barcodes)})"
            )
        return matrix, gene_names, barcodes

    # Load raw and filtered data
    print("Loading raw matrix...")
    raw_counts, raw_gene_names, raw_barcodes = extract_and_load_matrix(
        raw_tar_path, "raw"
    )
    print("Loading filtered matrix...")
    filtered_counts, filt_gene_names, filt_barcodes = extract_and_load_matrix(
        filtered_tar_path, "filtered"
    )

    # Verify gene names match
    if not np.array_equal(raw_gene_names, filt_gene_names):
        print("Warning: Gene names don't match between raw and filtered data")
        print(f"Raw genes: {len(raw_gene_names)}")
        print(f"Filtered genes: {len(filt_gene_names)}")
        # Try to find common genes
        common_genes = np.intersect1d(raw_gene_names, filt_gene_names)
        print(f"Common genes: {len(common_genes)}")
        if len(common_genes) > 0:
            # Subset to common genes
            raw_gene_idx = np.isin(raw_gene_names, common_genes)
            filt_gene_idx = np.isin(filt_gene_names, common_genes)
            raw_counts = raw_counts[raw_gene_idx, :]
            filtered_counts = filtered_counts[filt_gene_idx, :]
            raw_gene_names = raw_gene_names[raw_gene_idx]
            filt_gene_names = filt_gene_names[filt_gene_idx]
            print(f"Subsetted to {len(common_genes)} common genes")

    # Clean up
    import shutil

    shutil.rmtree(extract_dir)
    print("\nData loaded successfully:")
    print(f"Raw data: {raw_counts.shape} ({raw_counts.nnz:,} non-zero entries)")
    print(
        f"Filtered data: {filtered_counts.shape} ({filtered_counts.nnz:,} non-zero entries)"
    )

    return (
        raw_counts,
        filtered_counts,
        raw_gene_names,
        filt_gene_names,
        raw_barcodes,
        filt_barcodes,
    )


# Define the paths to your PBMC3k data files
raw_tar_path = "./pbmc3k_raw_gene_bc_matrices.tar.gz"
filtered_tar_path = "./pbmc3k_filtered_gene_bc_matrices.tar.gz"

# Load the data
raw_counts, filtered_counts, raw_genes, filt_genes, raw_barcodes, filt_barcodes = (
    load_10x_data_fixed(raw_tar_path, filtered_tar_path)
)

In [11]:
import scanpy as sc

# Create an AnnData object for further processing
adata = sc.AnnData(filtered_counts.T)  # Transpose to get cells x genes
adata.var["gene_names"] = filt_genes
adata.obs["barcodes"] = filt_barcodes

# Perform basic preprocessing
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var.highly_variable]

# Perform PCA
sc.tl.pca(adata, svd_solver="arpack")

# Compute the neighborhood graph
sc.pp.neighbors(adata)

# Perform UMAP for visualization
sc.tl.umap(adata)

# Perform clustering
sc.tl.leiden(adata)

# Extract clusters for comparison
clusters = adata.obs["leiden"].astype("category").cat.codes.values

# Prepare data dictionary for comparison
data_for_comparison = {
    "raw": raw_counts,
    "filtered": filtered_counts,
    "genes": filt_genes,
    "clusters": clusters,
    "contamination": 0.1,  # Example contamination level
}

C:\Users\nruff\PycharmProjects\decontx-python\.venv\Lib\site-packages\scanpy\preprocessing\_pca\__init__.py:385: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  adata.obsm[key_obsm] = X_pca
C:\Users\nruff\PycharmProjects\decontx-python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### actual comparison here:

!! only operates on HVG rn as it took forever - I think we gotta optimze a bit here and there...

In [14]:
py_contamination, r_contamination, py_time, r_time

(array([0.16875554, 0.19517861, 0.12340813, ..., 0.08460995, 0.12421349,
        0.14421421], shape=(2700,)),
 array([0.17634329, 0.1996392 , 0.13187085, ..., 0.09416792, 0.13424497,
        0.15401419], shape=(2700,)),
 111.91389489173889,
 20.11167049407959)